# 03 — Building Characteristics

Aggregates PLUTO building-level statistics per census tract.

**Data source:** PLUTO CSV (NYC only — `needs_pluto`).

**Output columns:** `tract_id`, `avg_floors`, `avg_yearbuilt`, `lot_area_mean`, `total_bldg_area`, `building_count`

**Output file:** `csv/03_building_characteristics.csv`

In [ ]:
ZONES_CONFIG = "zones.json"

In [ ]:
import pandas as pd
import numpy as np
import json
import os

os.makedirs("csv", exist_ok=True)

with open(ZONES_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

if not config["feature_flags"].get("needs_pluto", False):
    print("PLUTO not available — skipping notebook 03.")
    # Save empty CSV with correct columns
    df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
    df_empty = pd.DataFrame({"tract_id": df_tracts["tract_id"]})
    for col in ["avg_floors", "avg_yearbuilt", "lot_area_mean", "total_bldg_area", "building_count"]:
        df_empty[col] = np.nan
    df_empty.to_csv("csv/03_building_characteristics.csv", index=False)
    raise SystemExit("Skipped — needs_pluto=false")

PLUTO_PATH = config["pluto_path"]
BOROUGH_CODES = config["borough_codes"]
BOROUGH_FILTER = config["borough_filter"]
boro_code_filter = [str(BOROUGH_CODES[b]) for b in BOROUGH_FILTER]
print(f"Loading PLUTO from {PLUTO_PATH}")

In [ ]:
# ── Load PLUTO building columns ──────────────────────
COLS = ["borocode", "bct2020", "numfloors", "yearbuilt", "lotarea", "bldgarea", "numbldgs"]

df_pluto = pd.read_csv(PLUTO_PATH, usecols=COLS, dtype={"bct2020": str})
df_pluto = df_pluto[df_pluto["borocode"].astype(str).isin(boro_code_filter)].copy()

# Convert to numeric
for col in ["numfloors", "yearbuilt", "lotarea", "bldgarea", "numbldgs"]:
    df_pluto[col] = pd.to_numeric(df_pluto[col], errors="coerce")

# Filter out unreasonable values
df_pluto.loc[df_pluto["yearbuilt"] < 1700, "yearbuilt"] = np.nan
df_pluto.loc[df_pluto["numfloors"] <= 0, "numfloors"] = np.nan

print(f"PLUTO rows: {len(df_pluto):,}")

In [ ]:
# ── Aggregate per tract ───────────────────────────────
agg = df_pluto.groupby("bct2020").agg(
    avg_floors=("numfloors", "mean"),
    avg_yearbuilt=("yearbuilt", "mean"),
    lot_area_mean=("lotarea", "mean"),
    total_bldg_area=("bldgarea", "sum"),
    building_count=("numbldgs", "sum"),
).reset_index().rename(columns={"bct2020": "tract_id"})

# Round for readability
agg["avg_floors"] = agg["avg_floors"].round(1)
agg["avg_yearbuilt"] = agg["avg_yearbuilt"].round(0).astype("Int64")
agg["lot_area_mean"] = agg["lot_area_mean"].round(0)
agg["total_bldg_area"] = agg["total_bldg_area"].round(0)
agg["building_count"] = agg["building_count"].astype(int)

print(f"Aggregated {len(agg)} tracts")
print(agg.describe().round(1).to_string())

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/03_building_characteristics.csv"
agg.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(agg)} rows x {agg.shape[1]} cols)")
agg.head(10)